# ATOM3D-LBA Pocket Graph Regression with GNNVisualizer

This notebook compares graph-level **GCN**, **GraphSAGE**, **GAT**, and **GIN** regressors on the Hugging Face `vector-institute/atom3d-lba` dataset, selects the best model by validation MAE, then renders that best model with `GNNVisualizer`.

The task is ligand binding affinity regression: predict pK from a co-crystallized protein-ligand complex. The Hugging Face card exposes atomic numbers, 3D coordinates, pK labels, and token-type masks for protein, pocket, and ligand atoms. For visualization, the notebook keeps ligand atoms plus nearby pocket/protein atoms so the displayed graph stays around the 300-500 node range.

Source docs: [ATOM3D](https://www.atom3d.ai/), [ATOM3D-LBA Hugging Face dataset](https://huggingface.co/datasets/vector-institute/atom3d-lba), and the [GVP-GNN ATOM3D benchmark summary](https://icml-compbio.github.io/icml-website-2021/2021/papers/WCBICML2021_paper_15.pdf).

If imports fail in a fresh kernel, install the runtime packages first:

```bash
python3 -m pip install torch torch-geometric datasets
```

Optional environment variables: `ATOM3D_LBA_EPOCHS`, `ATOM3D_LBA_MAX_GRAPHS`, `ATOM3D_LBA_TARGET_NODES`, `ATOM3D_LBA_HIDDEN_CHANNELS`, and `ATOM3D_LBA_MODELS`.

`ATOM3D_LBA_MODELS` defaults to `GCN,GraphSAGE,GAT,GIN`.

In [ ]:
import os
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from IPython.display import Markdown, display
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, GCNConv, GINConv, SAGEConv, global_mean_pool

from gnn_exp import GNNVisualizer

In [ ]:
SEED = 7
torch.manual_seed(SEED)

EPOCHS = int(os.environ.get("ATOM3D_LBA_EPOCHS", "8"))
MAX_GRAPHS = int(os.environ.get("ATOM3D_LBA_MAX_GRAPHS", "96"))
TARGET_NODES = int(os.environ.get("ATOM3D_LBA_TARGET_NODES", "420"))
HIDDEN_CHANNELS = int(os.environ.get("ATOM3D_LBA_HIDDEN_CHANNELS", "16"))
BATCH_SIZE = int(os.environ.get("ATOM3D_LBA_BATCH_SIZE", "8"))
EDGE_RADIUS = float(os.environ.get("ATOM3D_LBA_EDGE_RADIUS", "4.5"))
KNN_EDGES = int(os.environ.get("ATOM3D_LBA_KNN_EDGES", "8"))
MODEL_NAMES = [
    name.strip()
    for name in os.environ.get("ATOM3D_LBA_MODELS", "GCN,GraphSAGE,GAT,GIN").split(",")
    if name.strip()
]
SUPPORTED_MODELS = {"GCN", "GraphSAGE", "GAT", "GIN"}
if any(name not in SUPPORTED_MODELS for name in MODEL_NAMES):
    raise ValueError(f"ATOM3D_LBA_MODELS must be drawn from {sorted(SUPPORTED_MODELS)}")
if HIDDEN_CHANNELS % 2 != 0:
    raise ValueError("ATOM3D_LBA_HIDDEN_CHANNELS must be divisible by 2 so GAT can use two heads")

COMMON_ATOMS = torch.tensor([1, 6, 7, 8, 9, 15, 16, 17, 35, 53], dtype=torch.long)


def nearest_to_ligand(indices, coords, ligand_coords, limit):
    if limit <= 0 or indices.numel() == 0:
        return indices[:0]
    distances = torch.cdist(coords[indices], ligand_coords).min(dim=1).values
    order = distances.argsort()
    return indices[order[:limit]]


def crop_atom_indices(coords, token_type, target_nodes):
    ligand_idx = (token_type == 2).nonzero(as_tuple=False).view(-1)
    pocket_idx = (token_type == 1).nonzero(as_tuple=False).view(-1)
    protein_idx = (token_type == 0).nonzero(as_tuple=False).view(-1)
    if ligand_idx.numel() == 0:
        ligand_idx = torch.arange(min(coords.size(0), max(1, target_nodes // 10)))
    ligand_coords = coords[ligand_idx]
    if ligand_idx.numel() > target_nodes:
        center = ligand_coords.mean(dim=0, keepdim=True)
        distances = torch.cdist(ligand_coords, center).view(-1)
        return ligand_idx[distances.argsort()[:target_nodes]]
    remaining = target_nodes - ligand_idx.numel()
    pocket_keep = nearest_to_ligand(pocket_idx, coords, ligand_coords, remaining)
    remaining -= pocket_keep.numel()
    protein_keep = nearest_to_ligand(protein_idx, coords, ligand_coords, remaining)
    keep = torch.cat([ligand_idx, pocket_keep, protein_keep]).unique(sorted=True)
    return keep


def build_edges(coords, radius=EDGE_RADIUS, k=KNN_EDGES):
    distances = torch.cdist(coords, coords)
    radius_mask = (distances <= radius) & (distances > 0)
    radius_edges = radius_mask.nonzero(as_tuple=False).t().contiguous()
    k = min(max(1, k), max(coords.size(0) - 1, 1))
    nearest = distances.topk(k + 1, largest=False).indices[:, 1:]
    sources = torch.arange(coords.size(0)).view(-1, 1).expand_as(nearest).reshape(-1)
    knn_edges = torch.stack([sources, nearest.reshape(-1)], dim=0)
    edge_index = torch.cat([radius_edges, knn_edges], dim=1)
    edge_index = torch.unique(edge_index, dim=1)
    reverse_edges = edge_index.flip(0)
    return torch.unique(torch.cat([edge_index, reverse_edges], dim=1), dim=1)


def atom_type_features(input_ids):
    known = (input_ids.view(-1, 1) == COMMON_ATOMS.view(1, -1)).float()
    other = (known.sum(dim=1, keepdim=True) == 0).float()
    return torch.cat([known, other], dim=1)


def atom_features(input_ids, token_type, coords, edge_index):
    centered = coords - coords.mean(dim=0, keepdim=True)
    scaled_coords = centered / centered.std(dim=0, keepdim=True).clamp_min(1.0)
    token_features = F.one_hot(token_type.clamp(0, 2), num_classes=3).float()
    ligand_coords = coords[token_type == 2]
    center = ligand_coords.mean(dim=0, keepdim=True) if ligand_coords.numel() else coords.mean(dim=0, keepdim=True)
    ligand_distance = torch.cdist(coords, center).view(-1, 1)
    ligand_distance = ligand_distance / ligand_distance.max().clamp_min(1.0)
    near_ligand = (ligand_distance < 0.25).float()
    degree = torch.bincount(edge_index[0], minlength=coords.size(0)).float().view(-1, 1)
    degree = degree / degree.max().clamp_min(1.0)
    return torch.cat([
        atom_type_features(input_ids),
        token_features,
        scaled_coords,
        ligand_distance,
        near_ligand,
        degree,
    ], dim=1)


def make_graph(example, target_nodes=TARGET_NODES):
    input_ids = torch.tensor(example["input_ids"], dtype=torch.long)
    coords = torch.tensor(example["coords"], dtype=torch.float32)
    token_type = torch.tensor(example["token_type_ids"], dtype=torch.long)
    keep = crop_atom_indices(coords, token_type, target_nodes)
    input_ids = input_ids[keep]
    coords = coords[keep]
    token_type = token_type[keep]
    edge_index = build_edges(coords)
    return Data(
        x=atom_features(input_ids, token_type, coords, edge_index),
        edge_index=edge_index,
        y=torch.tensor([float(example["labels"])], dtype=torch.float32),
        coords=coords,
        token_type=token_type,
    )


def split_graphs(graphs):
    order = torch.randperm(len(graphs)).tolist()
    shuffled = [graphs[index] for index in order]
    train_end = max(1, int(0.7 * len(shuffled)))
    val_end = max(train_end + 1, int(0.85 * len(shuffled))) if len(shuffled) > 2 else len(shuffled)
    return shuffled[:train_end], shuffled[train_end:val_end], shuffled[val_end:] or shuffled[train_end:val_end]


raw_dataset = load_dataset("vector-institute/atom3d-lba", split=f"train[:{MAX_GRAPHS}]")
graphs = [make_graph(raw_dataset[index]) for index in range(len(raw_dataset))]
train_graphs, val_graphs, test_graphs = split_graphs(graphs)
train_labels = torch.tensor([float(graph.y.item()) for graph in train_graphs])
all_labels = torch.tensor([float(graph.y.item()) for graph in graphs])
target_mean = train_labels.mean()
target_std = train_labels.std(unbiased=False).clamp_min(1e-6)
train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_graphs, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_graphs, batch_size=BATCH_SIZE)
visual_data = min(graphs, key=lambda data: abs(int(data.num_nodes) - TARGET_NODES))
query_pair = visual_data.edge_index[:, 0].tolist()
num_features = visual_data.x.size(1)

baseline_prediction = train_labels.mean()
baseline_val_mae = float(torch.tensor([abs(float(graph.y.item()) - baseline_prediction) for graph in val_graphs]).mean())
baseline_test_mae = float(torch.tensor([abs(float(graph.y.item()) - baseline_prediction) for graph in test_graphs]).mean())

display(Markdown(
    f"Loaded **ATOM3D-LBA** sample with {len(graphs)} cropped pocket graphs. "
    f"Split: {len(train_graphs)} train / {len(val_graphs)} val / {len(test_graphs)} test. "
    f"Visual graph: {visual_data.num_nodes} nodes, {visual_data.edge_index.size(1)} directed edges, "
    f"{num_features} visible node features. Label range: {all_labels.min():.2f}-{all_labels.max():.2f} pK. "
    f"Mean baseline MAE: {baseline_val_mae:.3f} val / {baseline_test_mae:.3f} test."
))

In [ ]:
class BaseLBARegressor(nn.Module):
    def pool_and_predict(self, x, batch):
        graph_embedding = global_mean_pool(x, batch)
        return self.regressor(graph_embedding)


class LBAGCN(BaseLBARegressor):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.act1 = nn.Tanh()
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.act2 = nn.Tanh()
        self.regressor = nn.Linear(hidden_channels, 1)

    def forward(self, x, edge_index, batch=None):
        x = x.float()
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        x = self.act1(self.conv1(x, edge_index))
        x = self.act2(self.conv2(x, edge_index))
        return self.pool_and_predict(x, batch)


class LBAGraphSAGE(BaseLBARegressor):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.act1 = nn.Tanh()
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.act2 = nn.Tanh()
        self.regressor = nn.Linear(hidden_channels, 1)

    def forward(self, x, edge_index, batch=None):
        x = x.float()
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        x = self.act1(self.conv1(x, edge_index))
        x = self.act2(self.conv2(x, edge_index))
        return self.pool_and_predict(x, batch)


class LBAGAT(BaseLBARegressor):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        heads = 2
        per_head_channels = hidden_channels // heads
        self.conv1 = GATConv(in_channels, per_head_channels, heads=heads, concat=True)
        self.act1 = nn.Tanh()
        self.conv2 = GATConv(hidden_channels, hidden_channels, heads=1, concat=False)
        self.act2 = nn.Tanh()
        self.regressor = nn.Linear(hidden_channels, 1)

    def forward(self, x, edge_index, batch=None):
        x = x.float()
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        x = self.act1(self.conv1(x, edge_index))
        x = self.act2(self.conv2(x, edge_index))
        return self.pool_and_predict(x, batch)


class LBAGIN(BaseLBARegressor):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.conv1 = GINConv(nn.Sequential(
            nn.Linear(in_channels, hidden_channels),
            nn.Tanh(),
            nn.Linear(hidden_channels, hidden_channels),
        ))
        self.act1 = nn.Tanh()
        self.conv2 = GINConv(nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.Tanh(),
            nn.Linear(hidden_channels, hidden_channels),
        ))
        self.act2 = nn.Tanh()
        self.regressor = nn.Linear(hidden_channels, 1)

    def forward(self, x, edge_index, batch=None):
        x = x.float()
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        x = self.act1(self.conv1(x, edge_index))
        x = self.act2(self.conv2(x, edge_index))
        return self.pool_and_predict(x, batch)


MODEL_BUILDERS = {
    "GCN": LBAGCN,
    "GraphSAGE": LBAGraphSAGE,
    "GAT": LBAGAT,
    "GIN": LBAGIN,
}

In [ ]:
def scaled_target(batch):
    return (batch.y.view(-1, 1).float() - target_mean) / target_std


def evaluate_mae(model, loader):
    model.eval()
    errors = []
    with torch.no_grad():
        for batch in loader:
            scaled_pred = model(batch.x, batch.edge_index, batch.batch)
            pred = scaled_pred * target_std + target_mean
            errors.append((pred.view(-1) - batch.y.view(-1).float()).abs())
    return float(torch.cat(errors).mean()) if errors else float("nan")


def train_model(model, train_loader, val_loader, epochs=EPOCHS):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.006, weight_decay=1e-4)
    best_state = {key: value.detach().clone() for key, value in model.state_dict().items()}
    best_val_mae = float("inf")
    best_epoch = 0
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            optimizer.zero_grad()
            pred = model(batch.x, batch.edge_index, batch.batch)
            loss = F.mse_loss(pred, scaled_target(batch))
            loss.backward()
            optimizer.step()
            total_loss += float(loss.detach()) * batch.num_graphs
        train_mse = total_loss / max(len(train_loader.dataset), 1)
        val_mae = evaluate_mae(model, val_loader)
        history.append({"epoch": epoch, "train_mse": train_mse, "val_mae": val_mae})
        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_epoch = epoch
            best_state = {key: value.detach().clone() for key, value in model.state_dict().items()}
    model.load_state_dict(best_state)
    return history, best_epoch, best_val_mae


def fit_candidate(name):
    torch.manual_seed(SEED)
    model = MODEL_BUILDERS[name](num_features, HIDDEN_CHANNELS)
    history, best_epoch, best_val_mae = train_model(model, train_loader, val_loader)
    train_mae = evaluate_mae(model, train_loader)
    test_mae = evaluate_mae(model, test_loader)
    return {
        "model": model,
        "history": history,
        "best_epoch": best_epoch,
        "train_mae": train_mae,
        "val_mae": best_val_mae,
        "test_mae": test_mae,
    }


results = {name: fit_candidate(name) for name in MODEL_NAMES}
best_model_name = min(results, key=lambda name: results[name]["val_mae"])
best_model = results[best_model_name]["model"]

rows = [
    "| Model | Best epoch | Train MAE pK | Val MAE pK | Test MAE pK |",
    "| --- | ---: | ---: | ---: | ---: |",
]
for name in MODEL_NAMES:
    result = results[name]
    marker = " **best**" if name == best_model_name else ""
    rows.append(
        f"| {name}{marker} | {result['best_epoch']} | {result['train_mae']:.3f} | "
        f"{result['val_mae']:.3f} | {result['test_mae']:.3f} |"
    )
rows.append(f"| Mean baseline | - | - | {baseline_val_mae:.3f} | {baseline_test_mae:.3f} |")
display(Markdown("\n".join(rows)))
display(Markdown(f"Selected **{best_model_name}** for visualization by lowest validation MAE."))

The next cell builds the widget for the best validation model. `renderer="auto"` keeps the visualization on WebGPU/WebGL when available.

In [ ]:
visualizer = GNNVisualizer(viewportHeight=1120, autoFit=True)
visualizer.add_model(
    data=visual_data,
    model=best_model.eval(),
    subgraphSample=False,
    queries=[query_pair],
    mode="graph",
)

EXPECTED_LAYER_TYPES = {
    "GCN": "GCNConv",
    "GraphSAGE": "SAGEConv",
    "GAT": "GATConv",
    "GIN": "GINConv",
}
EXPECTED_AGGREGATIONS = {
    "GCN": "gcn-normalized",
    "GraphSAGE": "mean",
    "GAT": "attention",
    "GIN": "sum",
}

assert visualizer.renderer == "auto"
assert visualizer.autoFit is True
assert visualizer.viewportHeight == 1120
assert visualizer.modelInfo["conv1"]["type"] == EXPECTED_LAYER_TYPES[best_model_name]
assert visualizer.modelInfo["conv1"].get("aggregation") == EXPECTED_AGGREGATIONS[best_model_name]
assert len(visualizer.graphData["x"]) == visual_data.num_nodes
assert "graphAggregation" in visualizer.intmData
assert len(visualizer.intmData["act1"][0]) == HIDDEN_CHANNELS

display(Markdown(
    "| Captured object | Value |\n"
    "| --- | ---: |\n"
    f"| Best model | {best_model_name} |\n"
    f"| Val MAE | {results[best_model_name]['val_mae']:.3f} pK |\n"
    f"| Test MAE | {results[best_model_name]['test_mae']:.3f} pK |\n"
    f"| Nodes | {len(visualizer.graphData['x'])} |\n"
    f"| Edges | {visual_data.edge_index.size(1)} |\n"
    f"| Hidden channels | {HIDDEN_CHANNELS} |\n"
    f"| Viewport height | {visualizer.viewportHeight}px |"
))

In [ ]:
display(visualizer)